In [1]:
from hybrid_ner import SchemaLoader, HybridNERPipeline
from hybrid_ner.models import Document
from hybrid_ner.generators import RegexCandidateGenerator

# full-test-pipeline-cell
import sys, re, os, json, pandas as pd

from hybrid_ner.schema import SchemaLoader, SchemaIndex
from hybrid_ner.generators import GazetteerGenerator, GazetteerConfig, NounPhraseGenerator
from hybrid_ner.generators.anchored_np_generator import AnchoredNPGenerator
from hybrid_ner.generators.description_embed_generator import DescriptionEmbedGenerator
from hybrid_ner.generators.candidate_utils import dedupe_candidate_spans
from hybrid_ner.pipeline import HybridNERPipeline
from hybrid_ner.models import Document, CandidateSpan
from hybrid_ner.classifier import SpanClassifier, TrainingExample

from hybrid_ner.llm_disambiguator import LLMDisambiguator, LLMConfig

In [2]:
# Check schema and library files
SCHEMA_PATH = "data/group-schema.json"           # your schema + descriptions
GAZ_PATH = "data/tag_keywords_lists.xlsx"       # your gazetteer

schema = SchemaLoader.load(SCHEMA_PATH)
print("Loaded schema labels:", len(schema.label_to_group))
print("Loaded groups:", len(schema.group_to_labels))

schema = json.load(open(SCHEMA_PATH))
schema_labels = set(schema.keys())

xl = pd.ExcelFile(GAZ_PATH)
excel_labels = set()

for sh in xl.sheet_names:
    df = xl.parse(sh)
    for col in df.columns:
        m = re.search(r"\[([^\]]+)\]\s*$", str(col))
        if m:
            excel_labels.add(m.group(1).strip())

missing_in_schema = sorted(excel_labels - schema_labels)
extra_in_schema = sorted(schema_labels - excel_labels)

print("Excel labels missing in schema:", missing_in_schema[:50], "… total", len(missing_in_schema))
print("Schema labels missing in excel:", extra_in_schema[:50], "… total", len(extra_in_schema))

print(os.getcwd())


Loaded schema labels: 39
Loaded groups: 10
Excel labels missing in schema: [] … total 0
Schema labels missing in excel: [] … total 0
/Users/mandd/Library/CloudStorage/OneDrive-IdahoNationalLaboratory/Desktop/NER-dackar


In [3]:
schema = SchemaLoader.load(SCHEMA_PATH)
print("Loaded schema labels:", len(schema.label_to_group))
print("Loaded groups:", len(schema.group_to_labels))
print(list(schema.group_to_labels.keys())[:10])

Loaded schema labels: 39
Loaded groups: 10
['G2_PHYSICAL_ASSET_SYSTEM', 'G10_ENV_BIO_CONTEXT', 'G3_MATERIAL_CHEMISTRY', 'G1_PHYSICAL_COMPONENT', 'G4_MECHANISM_PROCESS', 'G9_DIAGNOSTIC_QUAL', 'G5_FAILURE_OUTCOME', 'G6_ACTION_MAINT_SURV', 'G7_TOOL_METHOD', 'G8_PROPERTY_STATE']


In [4]:
import requests

r = requests.post(
    "http://localhost:11434/v1/chat/completions",
    json={
        "model": "llama3.2:latest",
        "messages": [{"role": "user", "content": "Reply with exactly OK."}],
        "temperature": 0,
    },
    timeout=10,
)
print(r.status_code)
print(r.json()["choices"][0]["message"]["content"])

200
OK.


In [5]:
# 1) load schema (fallback minimal)
try:
    schema = SchemaLoader.load(SCHEMA_PATH)
except Exception:
    label_to_group = {
        "deg_mech": "G5_MECHANISMS",
        "comp_mech_spec": "G1_PHYSICAL",
        "fail_type_n": "G6_OUTCOMES_OCCURRENCES",
        "event": "G6_OUTCOMES_OCCURRENCES",
    }
    group_to_labels = {}
    for lbl, grp in label_to_group.items():
        group_to_labels.setdefault(grp, set()).add(lbl)
    schema = SchemaIndex(label_to_group=label_to_group, group_to_labels=group_to_labels, pair_decision={}, conditional_rules=[])

# 1b) Initialize LLM (AFTER schema is defined)
llm_cfg = LLMConfig(
    use_cli=False,
    http_url="http://localhost:11434/v1/chat/completions",
    model="llama3.2:latest",
    temperature=0.0,
    timeout=10,
    dry_run=False)
llm = LLMDisambiguator(schema=schema, config=llm_cfg)

# 2) generators
gaz = GazetteerGenerator(GAZ_PATH, config=GazetteerConfig(
    match_mode="fuzzy_tokens", fuzzy_jaccard_threshold=0.75, max_window_tokens=8, emit_overlapping=True))
np_base = NounPhraseGenerator()

# build token->(label,term) for PHYSICAL components (anchored high-precision)
token_index = {}
for lbl, terms in gaz.label_terms.items():
    if schema.label_to_group.get(lbl) != "G1_PHYSICAL_COMPONENT":
        continue
    for term in terms:
        for tok in re.findall(r"\w+", term.lower()):
            if len(tok) < 3:
                continue
            # optional denylist to avoid adjectives ("adhesive","wear","acid")
            if tok in {"adhesive","acid","wear","attack","degradation","pitting"}:
                continue
            token_index.setdefault(tok, set()).add((lbl, term))

anchored = AnchoredNPGenerator(np_base, token_index, max_tokens=4)

# Build token index for MECHANISMS (so anchors pick up mechanism-centered NPs)
mech_token_index = {}
for lbl, terms in gaz.label_terms.items():
    if schema.label_to_group.get(lbl) != "G4_MECHANISM_PROCESS":
        continue
    for term in terms:
        for tok in re.findall(r"\w+", term.lower()):
            if len(tok) < 3:
                continue
            mech_token_index.setdefault(tok, set()).add((lbl, term))

mech_anchored = AnchoredNPGenerator(np_base, mech_token_index, max_tokens=4)

# 3) description-embed generator (uses same group-schema.json if it contains descriptions)
desc_gen = DescriptionEmbedGenerator(
    label_json_path=SCHEMA_PATH,
    gazetteer_path=GAZ_PATH,
    score_threshold=0.30,    # start lower while validating
    sim_floor=0.00,
)
# fit will build embeddings and lexical cues (requires sentence-transformers or tfidf fallback)
desc_gen.fit(schema)

# 4) small classifier placeholder (optional). Keep very light — the pipeline will still use gaz + desc_gen
clf = SpanClassifier(min_prob=0.25)
toy_examples = [
    TrainingExample(doc_text="Adhesive wear on the bearing", start=0, end=len("Adhesive wear"), label="deg_mech"),
    TrainingExample(doc_text="Adhesive wear on the bearing", start=18, end=18+len("bearing"), label="comp_mech_spec"),
    TrainingExample(doc_text="Seal degradation observed", start=0, end=len("Seal degradation"), label="deg_mech"),
]
clf.fit(toy_examples, schema)

# 5) Build pipeline (order matters: gazetteer -> anchored NP -> desc_embed -> classifier)
pipe = HybridNERPipeline(schema=schema, 
                         generators=[gaz, anchored, mech_anchored], 
                         desc_gen=desc_gen,
                         classifier=clf,
                         llm_disambiguator=llm)

# 6) convenience: allow compatibility engine access to token index if it expects it
if hasattr(pipe, "compatibility"):
    pipe.compatibility.token_index = token_index

# 7) test text
text = ("Work order: Inspection found adhesive wear on the bearing and evidence of acid attack on the gasket. "
        "Additional notes mention seal degradation and pitting damage near the cam shaft region.")
doc = Document(doc_id="DOC_FULL_TEST", text=text)

# 8) run
res = pipe.run(doc)

print("=== Final Pipeline Entities ===")
for e in res.entities:
    print(f"- '{e.text}' labels={e.labels} groups={e.groups} [{e.start},{e.end}]")

print("\n=== Decisions ===")
for d in res.decisions:
    outs = [(s.text, s.labels) for s in d.output_spans]
    print(f"- action={d.action} input={d.input_span_ids} out={outs} notes={d.notes}")

# 9) debug: show candidates before decisions (if pipeline exposes them)
# (many pipelines keep candidates internal, but if your pipeline returns intermediate traces you can print them)
if hasattr(res, "candidates"):
    print("\n=== Raw candidates ===")
    for c in res.candidates:
        print(c.span_id, c.text, c.sources)


=== Final Pipeline Entities ===
- 'adhesive wear' labels=['deg_mech'] groups=['G4_MECHANISM_PROCESS'] [29,42]
- 'bearing' labels=['comp_mech_rot'] groups=['G1_PHYSICAL_COMPONENT'] [50,57]
- 'acid' labels=['ast_I_C'] groups=['G2_PHYSICAL_ASSET_SYSTEM'] [74,78]
- 'acid attack' labels=['deg_mech'] groups=['G4_MECHANISM_PROCESS'] [74,85]
- 'gasket' labels=['comp_mech_spec'] groups=['G1_PHYSICAL_COMPONENT'] [93,99]
- 'seal' labels=['comp_mech_struct'] groups=['G1_PHYSICAL_COMPONENT'] [126,130]
- 'seal degradation' labels=['ast_mech'] groups=['G2_PHYSICAL_ASSET_SYSTEM'] [126,142]
- 'degradation' labels=['deg_mech'] groups=['G4_MECHANISM_PROCESS'] [131,142]
- 'damage' labels=['fail_type_v'] groups=['G5_FAILURE_OUTCOME'] [155,161]
- 'cam shaft' labels=['comp_mech_rot'] groups=['G1_PHYSICAL_COMPONENT'] [171,180]
- 'shaft' labels=['ast_eln'] groups=['G2_PHYSICAL_ASSET_SYSTEM'] [175,180]

=== Decisions ===
- action=accept input=['5cb5dc1b-df6d-4bdc-9cef-0d6ae4eaf727'] out=[('gasket', ['comp_mech_

In [6]:
print("Any LLM rationale attached?",
      any(h.rationale and "llm" not in (h.rationale or "").lower()
          for c in getattr(res, "candidates", []) for h in getattr(c, "proposed_labels", [])))


Any LLM rationale attached? False
